# 🧠 Tutorial Completo: Fashion-MNIST (Zalando Research)

Este cuaderno es una guía paso a paso para trabajar con el dataset **Fashion-MNIST**, creado por Zalando Research como un reemplazo directo del clásico MNIST.

## 🎯 Objetivos del Tutorial
1. Cargar y explorar el dataset.
2. Preprocesar las imágenes.
3. Construir y entrenar una red neuronal con TensorFlow/Keras.
4. Evaluar el rendimiento del modelo.
5. **Simulador interactivo**: Probar el modelo con datos reales y un simulador en HTML.

## 📦 Paso 1: Configuración e Importación de Librerías

Importamos las herramientas necesarias para el manejo de datos, construcción del modelo y visualización.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import io
import base64

# Configuración para visualizaciones
plt.rcParams['figure.figsize'] = (10, 6)
print(f"TensorFlow versión: {tf.__version__}")

## 📊 Paso 2: Carga y Exploración del Dataset

Fashion-MNIST contiene 70,000 imágenes en escala de grises de 28x28 píxeles, divididas en 60,000 para entrenamiento y 10,000 para prueba.

### Etiquetas (Clases):
0: T-shirt/top, 1: Trouser, 2: Pullover, 3: Dress, 4: Coat,
5: Sandal, 6: Shirt, 7: Sneaker, 8: Bag, 9: Ankle boot

In [ ]:
# Cargar el dataset directamente desde Keras (descarga automática)
fashion_mnist = tf.keras.datasets.fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Nombres de las clases
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Forma de x_train: {x_train.shape}")
print(f"Forma de y_train: {y_train.shape}")
print(f"Forma de x_test: {x_test.shape}")
print(f"Forma de y_test: {y_test.shape}")

## 👁️ Paso 3: Visualización de Muestras

Visualicemos las primeras 10 imágenes del conjunto de entrenamiento para entender cómo se ven los datos.

In [ ]:
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i], cmap=plt.cm.binary)
    plt.xlabel(class_names[y_train[i]])
plt.tight_layout()
plt.show()

## ⚙️ Paso 4: Preprocesamiento de Datos

Las imágenes tienen valores de píxeles entre 0 y 255. Para que la red neuronal entrene de manera eficiente, normalizamos estos valores al rango [0, 1].

In [ ]:
# Normalizar las imágenes
x_train = x_train / 255.0
x_test = x_test / 255.0

print(f"Valor mínimo: {x_train.min()}, Valor máximo: {x_train.max()}")

## 🏗️ Paso 5: Construcción del Modelo

Crearemos una red neuronal secuencial con:
1. **Flatten**: Convierte la matriz 28x28 en un vector de 784 elementos.
2. **Dense (128)**: Capa oculta con 128 neuronas y activación ReLU.
3. **Dropout (0.2)**: Regularización para evitar sobreajuste (overfitting).
4. **Dense (10)**: Capa de salida con 10 neuronas (una por cada clase).

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.summary()

## 🚀 Paso 6: Compilación y Entrenamiento

Compilamos el modelo con el optimizador 'adam', la función de pérdida 'sparse_categorical_crossentropy' (ya que las etiquetas son enteros) y monitoreamos la 'accuracy'.

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Entrenar el modelo
history = model.fit(x_train, y_train, epochs=10, 
                    validation_split=0.1, verbose=1)

## 📈 Paso 7: Evaluación del Modelo

Evaluamos el rendimiento en el conjunto de prueba y graficamos la curva de aprendizaje.

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f'\nPrecisión en el conjunto de prueba: {test_acc:.4f}')

# Graficar historial de entrenamiento
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Precisión del Modelo')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Pérdida del Modelo')
plt.show()

## 🎮 Paso 8: Simulador Interactivo en el Cuaderno

Usa este panel interactivo para seleccionar una imagen del conjunto de prueba y ver la predicción del modelo junto con las probabilidades de cada clase.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Crear widgets
index_slider = widgets.IntSlider(min=0, max=len(x_test)-1, step=1, description='Índice:', continuous_update=False)
predict_button = widgets.Button(description='Predecir', button_style='success')
output_area = widgets.Output()

# Función de predicción
def on_predict_click(b):
    with output_area:
        clear_output()
        idx = index_slider.value
        img = x_test[idx]
        true_label = class_names[y_test[idx]]
        
        # Expandir dimensiones para el modelo (1, 28, 28)
        img_array = np.expand_dims(img, axis=0)
        predictions = model.predict(img_array)[0]
        predicted_label = class_names[np.argmax(predictions)]
        
        # Mostrar imagen
        plt.figure(figsize=(4, 4))
        plt.imshow(img, cmap=plt.cm.binary)
        plt.title(f"Real: {true_label}\nPredicho: {predicted_label}", 
                  color='green' if true_label == predicted_label else 'red')
        plt.axis('off')
        plt.show()
        
        # Mostrar barras de probabilidad
        plt.figure(figsize=(8, 4))
        plt.barh(class_names, predictions, color='skyblue')
        plt.xlabel('Probabilidad')
        plt.title('Probabilidades por Clase')
        plt.xlim(0, 1)
        plt.gca().invert_yaxis()
        plt.show()

predict_button.on_click(on_predict_click)

print("Selecciona un índice y presiona 'Predecir'")
display(index_slider, predict_button, output_area)

# Ejecutar una predicción inicial
on_predict_click(None)

## 🌐 Paso 9: Simulador en HTML (Canvas de Dibujo)

A continuación se proporciona un **código HTML/JS completo** que se guardará como `simulador.html` y podrás abrir en tu navegador. 

Este simulador te permite **dibujar** una prenda con el mouse y ver cómo un modelo (preparado para TensorFlow.js) la clasificaría. 

> **Nota:** Para que la predicción funcione con tu modelo real en el navegador, deberías exportar tu modelo de Keras a TensorFlow.js (`model.save('my_model')` y convertirlo con `tensorflowjs_converter`). Este HTML está listo como plantilla funcional para recibir esa integración.

In [ ]:
html_code = '''<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Simulador Fashion-MNIST</title>
    <style>
        body { font-family: Arial, sans-serif; text-align: center; background: #f4f4f9; padding: 20px; }
        canvas { border: 2px solid #333; background: black; cursor: crosshair; touch-action: none; }
        .controls { margin-top: 15px; }
        button { padding: 10px 20px; font-size: 16px; margin: 5px; cursor: pointer; border: none; border-radius: 5px; }
        .btn-clear { background: #e74c3c; color: white; }
        .btn-predict { background: #2ecc71; color: white; }
        #result { margin-top: 20px; font-size: 18px; font-weight: bold; color: #2c3e50; }
    </style>
</head>
<body>
    <h2>🎨 Dibuja una prenda (28x28)</h2>
    <canvas id="drawCanvas" width="280" height="280"></canvas>
    <div class="controls">
        <button class="btn-clear" onclick="clearCanvas()">Limpiar</button>
        <button class="btn-predict" onclick="simulatePrediction()">Simular Predicción</button>
    </div>
    <div id="result">Esperando dibujo...</div>

    <script>
        const canvas = document.getElementById('drawCanvas');
        const ctx = canvas.getContext('2d');
        let isDrawing = false;

        // Inicializar canvas en negro (como el dataset)
        ctx.fillStyle = 'black';
        ctx.fillRect(0, 0, canvas.width, canvas.height);
        ctx.strokeStyle = 'white';
        ctx.lineWidth = 15;
        ctx.lineCap = 'round';

        function getPos(e) {
            const rect = canvas.getBoundingClientRect();
            const clientX = e.touches ? e.touches[0].clientX : e.clientX;
            const clientY = e.touches ? e.touches[0].clientY : e.clientY;
            return { x: clientX - rect.left, y: clientY - rect.top };
        }

        canvas.addEventListener('mousedown', (e) => { isDrawing = true; draw(e); });
        canvas.addEventListener('mousemove', (e) => { if (isDrawing) draw(e); });
        canvas.addEventListener('mouseup', () => { isDrawing = false; ctx.beginPath(); });
        canvas.addEventListener('touchstart', (e) => { isDrawing = true; draw(e); e.preventDefault(); });
        canvas.addEventListener('touchmove', (e) => { if (isDrawing) draw(e); e.preventDefault(); });
        canvas.addEventListener('touchend', () => { isDrawing = false; ctx.beginPath(); });

        function draw(e) {
            const pos = getPos(e);
            ctx.lineTo(pos.x, pos.y);
            ctx.stroke();
            ctx.beginPath();
            ctx.moveTo(pos.x, pos.y);
        }

        function clearCanvas() {
            ctx.fillStyle = 'black';
            ctx.fillRect(0, 0, canvas.width, canvas.height);
            document.getElementById('result').innerText = 'Esperando dibujo...';
        }

        function simulatePrediction() {
            // Simulación de predicción (en un caso real, aquí se pasaría el canvas a un modelo TF.js)
            const classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'];
            const randomClass = classes[Math.floor(Math.random() * classes.length)];
            const confidence = (Math.random() * 0.4 + 0.6).toFixed(2); // Entre 60% y 100%
            document.getElementById('result').innerText = `Predicción simulada: ${randomClass} (${(confidence*100).toFixed(0)}% de confianza)`;
        }
    </script>
</body>
</html>'''

with open('simulador_fashion_mnist.html', 'w') as f:
    f.write(html_code)

print("✅ Archivo 'simulador_fashion_mnist.html' creado exitosamente.")
print("Puedes descargarlo desde la barra lateral de archivos de Colab (ícono de carpeta) y abrirlo en tu navegador.")

# Descomenta la siguiente línea para descargarlo automáticamente en Colab
from google.colab import files
files.download('simulador_fashion_mnist.html')